# SASRec 全量训练 — B7 最优超参（maxlen=100）

来源：团队网格搜索任务 **B7**（王彬哲），见 [`results/grid_search/maxlen100_team_B_ranking.md`](../results/grid_search/maxlen100_team_B_ranking.md)。

数据目录：`SASRec/data/`。请先从 GitHub Release 准备数据，见 `数据与模型下载.md` 或 `python scripts/download_release_assets.py`。

**要求**：推荐 **GPU** + `batch_size=512`。全量约 98.6 万用户评估，预计 **7～11 小时**（视显卡而定）。

## B7 超参（FAST 网格 valid NDCG@10 ≈ 0.7945）

| 参数 | 值 |
|------|-----|
| maxlen | 100 |
| hidden_units | 128 |
| num_blocks | 3 |
| num_heads | 4 |
| dropout_rate | 0.15 |
| lr | 1e-3 |
| batch_size | 512（GPU） |
| num_epochs | 8（GPU） |
| eval_user_limit | None（全量） |
| eval_num_neg | 200 |

通用全量训练见 `01_full_train.ipynb`（默认 dropout=0.2）。本 notebook 单独保存为 `data/sasrec_full_b7_memmap.pt`。

In [1]:
from pathlib import Path
import sys

def resolve_sasrec_dir() -> Path:
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd / "SASRec", cwd.parent / "SASRec"):
        if (c / "data").is_dir() and (c / "scripts").is_dir():
            return c
    raise FileNotFoundError("Cannot locate SASRec/ (need data/ and scripts/).")

SASREC_DIR = resolve_sasrec_dir()
REPO_ROOT = SASREC_DIR.parent
if str(SASREC_DIR) not in sys.path:
    sys.path.insert(0, str(SASREC_DIR))
assert (SASREC_DIR / "sasrec_core").is_dir(), "缺少 SASRec/sasrec_core/"

cache_dir = SASREC_DIR / "data"
print("SASREC_DIR:", SASREC_DIR)
print("cache_dir:", cache_dir)

SASREC_DIR: E:\oytproject\UserBehavior_Analysis\SASRec
cache_dir: E:\oytproject\UserBehavior_Analysis\SASRec\data


In [2]:
from __future__ import annotations

import json
import shutil
from datetime import datetime

import pandas as pd
import torch

from sasrec_core import (
    SASRecConfig,
    SASRecEstimator,
    build_memmap_cache,
    load_memmap_meta,
)

split_train_path = cache_dir / "train.parquet"
split_valid_path = cache_dir / "valid.parquet"
split_test_path = cache_dir / "test.parquet"
item_map_path = cache_dir / "item2idx_mapping.parquet"

memmap_dir = cache_dir / "memmap_cache"
model_path = cache_dir / "sasrec_full_b7_memmap.pt"
metrics_path = cache_dir / "baseline" / "sasrec_full_b7_metrics.json"

# 设为 True 会删除 memmap 与 B7 模型后重训（不删除 01 默认的 sasrec_full_memmap.pt）
cleanup_old_artifacts = True

required_files = [split_train_path, split_valid_path, split_test_path, item_map_path]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"缺少训练输入文件: {missing}")

if cleanup_old_artifacts:
    if memmap_dir.exists():
        shutil.rmtree(memmap_dir)
        print("已删除旧 memmap:", memmap_dir)
    if model_path.exists():
        model_path.unlink()
        print("已删除旧 B7 模型:", model_path)

train_rows = int(pd.read_parquet(split_train_path, columns=["user_id"]).shape[0])
print("train 用户数:", train_rows)
print("模型将保存至:", model_path)

已删除旧 memmap: E:\oytproject\UserBehavior_Analysis\SASRec\data\memmap_cache
train 用户数: 985877
模型将保存至: E:\oytproject\UserBehavior_Analysis\SASRec\data\sasrec_full_b7_memmap.pt


In [3]:
memmap_dir = build_memmap_cache(cache_dir=cache_dir, overwrite=True)
meta = load_memmap_meta(memmap_dir)

item_map_df = pd.read_parquet(item_map_path, columns=["item_id", "item_idx"])
idx2item = dict(zip(item_map_df["item_idx"].astype(int), item_map_df["item_id"].astype(int)))

print("memmap:", memmap_dir)
print("用户数:", meta["num_users"])
print("交互数:", meta["num_interactions"])
print("物品数:", meta["itemnum"])

memmap: E:\oytproject\UserBehavior_Analysis\SASRec\data\memmap_cache
用户数: 985877
交互数: 92620273
物品数: 1795765


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
is_cuda = device == "cuda"
if not is_cuda:
    print("警告: 未检测到 CUDA，B7 全量训练强烈建议使用 GPU。")

# B7 网格最优（maxlen=100）；CPU 时自动降级
B7_MAXLEN = 100
B7_HIDDEN = 128 if is_cuda else 64
B7_BLOCKS = 3 if is_cuda else 2
B7_HEADS = 4 if is_cuda else 2
B7_DROPOUT = 0.15
B7_LR = 1e-3
B7_BATCH = 512 if is_cuda else 256
B7_EPOCHS = 8 if is_cuda else 3

eval_user_limit = None  # 全量约 98.6 万用户
eval_num_neg = 200

config = SASRecConfig(
    maxlen=B7_MAXLEN,
    hidden_units=B7_HIDDEN,
    num_blocks=B7_BLOCKS,
    num_heads=B7_HEADS,
    dropout_rate=B7_DROPOUT,
    batch_size=B7_BATCH,
    lr=B7_LR,
    num_epochs=B7_EPOCHS,
    num_workers=0,
    eval_num_neg=eval_num_neg,
    eval_k=10,
    seed=42,
)

print("device:", device)
print("task: B7 (王彬哲 grid search)")
print(config)

est = SASRecEstimator(config=config, device=device)

警告: 未检测到 CUDA，B7 全量训练强烈建议使用 GPU。
device: cpu
task: B7 (王彬哲 grid search)
SASRecConfig(maxlen=100, hidden_units=64, num_blocks=2, num_heads=2, dropout_rate=0.15, batch_size=256, lr=0.001, betas=(0.9, 0.98), weight_decay=0.0, grad_clip_norm=5.0, num_epochs=3, num_workers=0, eval_num_neg=200, eval_k=10, seed=42)


In [ ]:
est.fit(
    input_mode="memmap",
    cache_dir=cache_dir,
    memmap_dir=memmap_dir,
    itemnum=int(meta["itemnum"]),
    idx2item=idx2item,
    rebuild_memmap_cache=False,
    eval_user_limit=eval_user_limit,
    verbose=True,
)

In [ ]:
valid_metrics = est.evaluate(mode="valid", eval_user_limit=eval_user_limit)
test_metrics = est.evaluate(mode="test", eval_user_limit=eval_user_limit)

print("验证集:", valid_metrics)
print("测试集:", test_metrics)

history_df = pd.DataFrame(est.history)
history_df

In [ ]:
est.save(model_path, include_data=False)
print("模型已保存:", model_path)

metrics_path.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "task_id": "B7",
    "member": "王彬哲",
    "created_at": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "source_model": model_path.name,
    "hyperparams": {
        "maxlen": B7_MAXLEN,
        "hidden_units": B7_HIDDEN,
        "num_blocks": B7_BLOCKS,
        "num_heads": B7_HEADS,
        "dropout_rate": B7_DROPOUT,
        "lr": B7_LR,
        "batch_size": B7_BATCH,
        "num_epochs": B7_EPOCHS,
        "eval_num_neg": eval_num_neg,
        "eval_user_limit": eval_user_limit,
    },
    "metrics": {"valid": valid_metrics, "test": test_metrics},
    "notes": "Full train with B7 grid-search winner (maxlen=100). Compare with baseline_sasrec_20260425_130830.json.",
}
metrics_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("指标已保存:", metrics_path)

demo_user = int(pd.read_parquet(split_train_path, columns=["user_id"]).iloc[0, 0])
topk = est.recommend(user_idx=demo_user, k=10, chunk_size=20000)
print("示例用户:", demo_user)
print("Top10:", topk)